In [1]:
import warnings
warnings.filterwarnings('ignore')
import os
import scanpy as sc
import pandas as pd
import squidpy as sq
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np

In [2]:
in_dir = "../../results/03_kidney_atlas/06_run_cell2location"
out_dir = '../../results/03_kidney_atlas/07_viz_cell2location'

os.makedirs(out_dir, exist_ok=True)

In [3]:
sample_list = ['15837', '15852', '16512', '21022', '8262', 
               'B23-03794-5', 'B23-05785-7-1', 'B23-05785-7-2', 
               'X1842', 'X1845', 'X1899', 'X1912']

In [4]:
for sample in sample_list:
    print(f'Processing sample {sample} ...')
    for subclass in ['l1', 'l2']:
        os.makedirs(f'{out_dir}/subclass_{subclass}/{sample}', exist_ok=True)

        adata = sc.read_h5ad(f'{in_dir}/{sample}_subclass_{subclass}.h5ad')
        adata = adata[adata.obs['in_tissue'] == 1, ].copy()
        
        sq.gr.spatial_neighbors(
            adata,
            coord_type="grid",
            n_neighs=6,      
        )

        adj = adata.obsp['spatial_connectivities']  # sparse matrix (n_spots x n_spots)

        # Count number of neighbors per spot
        num_neighbors = np.array(adj.sum(axis=1)).flatten()
        
        # Spots with at least 1 neighbor
        keep_mask = num_neighbors > 2

        adata = adata[keep_mask].copy()

        for cell_type in adata.uns['mod']['factor_names'].tolist():
            fig, axes = plt.subplots(2, 1, figsize=(10, 5))
            sc.pl.spatial(adata, color=cell_type, cmap='magma', size=0, img_key='hires', alpha_img=1, 
                        vmax='p99', ax=axes[0], show=False)
            sc.pl.spatial(adata, color=cell_type, cmap='magma', size=1, img_key='hires', alpha_img=1, 
                        vmax='p99', ax=axes[1], show=False)

            # make sure axes have the same size
            axes[0].set_aspect('equal')
            axes[1].set_aspect('equal')
            axes[0].set_xlabel('')
            axes[0].set_ylabel('')
            axes[1].set_xlabel('')
            axes[1].set_ylabel('')

            fig.savefig(f'{out_dir}/subclass_{subclass}/{sample}/{cell_type}.png', dpi=300, bbox_inches='tight')
            fig.tight_layout()
            plt.close()

        adata.write_h5ad(f'{out_dir}/subclass_{subclass}/{sample}.h5ad')

Processing sample 15837 ...
Processing sample 15852 ...
Processing sample 16512 ...
Processing sample 21022 ...
Processing sample 8262 ...
Processing sample B23-03794-5 ...
Processing sample B23-05785-7-1 ...
Processing sample B23-05785-7-2 ...
Processing sample X1842 ...
Processing sample X1845 ...
Processing sample X1899 ...
Processing sample X1912 ...
